# Next-N-Minutes ETH Price, Order-Flow & Flash-Crash Forecaster (Colab Pro GPU Suite)

This self-contained notebook unpacks the cached **31,283 real Ethereum Mainnet Uniswap V3 swaps**, **176 `TickLens` liquidity-book snapshots**, and **Aave V3 borrower positions** (`data/mainnet_swaps.json`, `data/mainnet_checkpoints.json`, `data/aave_v3_borrowers.json`) and runs the 4-part **ETH Price & Order-Flow Forecasting Suite**:

1. **Experiment 1 — Multi-Horizon Unconditional Order-Flow & ETH Price Forecasting (`H = 10, 25, 50, 100` swaps / `~2.5 to 25 min` ahead):**
   - Compares `LastPrice_Naive`, `Momentum_Reversion_EWMA`, `OrderFlowMLPForecaster` (time-series only), `OrderFlowTransformerForecaster` (32-trade attention), and **`LiquidityCoupledWorldForecaster`** (order-flow sequence predictor + vectorized 32-bin `TickLens` liquidity-book price-impact integrator).
2. **Experiment 2 — Flow-Conditioned ETH Price Path Forecasting:**
   - Evaluates how accurately each architecture translates incoming order flow (both predicted and realized) across the live Uniswap V3 `TickLens` liquidity book into future ETH/USD prices.
3. **Experiment 3 — Pretrained `Qwen2.5` (`0.5B`, `1.5B`, `3B`) LoRA/QLoRA Price & Order-Flow Forecasters:**
   - Fine-tunes pretrained `Qwen2.5` foundation models end-to-end on tokenized 32-trade order-flow history + 32-bin `TickLens` spatial liquidity tokens.
4. **Experiment 4 — Fee-Aware DeFi Trading Strategy (`$100,000` Notional), Smart Execution Timing, & Air-Pocket Crash Early Warning.**

**Google Drive Live Checkpointing:** Every experiment saves atomically to `/content/drive/MyDrive/evm_intervene_forecast/` after each model finishes, so no progress is ever lost if your Colab session disconnects.

In [ ]:
# Cell 1: Mount Google Drive, Unpack Self-Contained Repo + Mainnet Data (`data/*.json`), and Install Dependencies
import base64, io, os, subprocess, sys, tarfile, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Please enable GPU in Colab: Runtime -> Change runtime type -> L4 / A100 / T4 GPU'
print(f'[gpu] {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = '/content/drive/MyDrive/evm_intervene_forecast'
    print(f'[drive] Mounted! Live outputs will be saved to: {OUT_DIR}')
except Exception as e:
    OUT_DIR = '/content/evm-intervene/results/forecast_suite'
    print(f'[drive] Running without Google Drive mount ({e}); saving to: {OUT_DIR}')

os.makedirs(OUT_DIR, exist_ok=True)
os.environ['OUT_DIR'] = OUT_DIR

REPO_TAR_XZ_B64 = '...'
os.makedirs('/content/evm-intervene', exist_ok=True)
os.chdir('/content/evm-intervene')
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(REPO_TAR_XZ_B64)), mode='r:xz') as tf:
    tf.extractall('.')
print('[ok] Unpacked code + Mainnet data into /content/evm-intervene:', sorted(os.listdir('data')))

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'accelerate', 'bitsandbytes', 'peft', 'transformers', 'scipy'], check=True)
print('[ok] Environment ready!')


In [ ]:
# Cell 2: Verify Mainnet Data Files & Existing Google Drive Checkpoints
import os, glob
print('Mainnet data files:', sorted(glob.glob('/content/evm-intervene/data/*.json')))
print('Existing checkpoints in Google Drive:', sorted(glob.glob(os.environ['OUT_DIR'] + '/*.json')))


In [ ]:
# Cell 3: Run Full ETH Price, Order-Flow & Qwen2.5 LoRA Forecasting Suite (auto-saves to Google Drive)
import os, subprocess, sys
out_dir = os.environ['OUT_DIR']
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run([sys.executable, '-u', 'scripts/run_colab_forecaster_suite.py', '--out-dir', out_dir], env=env, check=True)


In [ ]:
# Cell 4: Inspect Saved Results & Download JSON Artifacts
import os
from pathlib import Path

out_dir = Path(os.environ['OUT_DIR'])
for p in sorted(out_dir.glob('*.json')):
    print(f'=== {p.name} ({p.stat().st_size:,} bytes) ===')

try:
    from google.colab import files
    for p in sorted(out_dir.glob('*.json')):
        files.download(str(p))
except Exception as e:
    print('Files are safely stored in Google Drive at:', out_dir)
